In [2]:
import pandas as pd
import numpy as np

In [ ]:
ripple_4k7_100n_12 = 140e-3
ripple_10k_100n_12 = 65.65e-3
ripple_4k7_100n_37 = 45.9e-3
ripple_10k_100n_37 = 21.58e-3

In [5]:
tau_4k7_100n = (4.7e3 * 100e-9)
tau_10k_100n = (10e3 * 100e-9)
print(tau_4k7_100n, tau_10k_100n)

0.00047 0.001


In [9]:
data = {
    "Wrap": [4000, 8000, 12000, 16000],
}
df = pd.DataFrame(data)
df["Freq[Hz]"] = 150e6 / df["Wrap"]
df["Ripple_10k_100n"] = 3.3 * (1 - np.exp(- 1 / (df["Freq[Hz]"] * tau_10k_100n)))
df

,Wrap,Freq[Hz],Ripple_10k_100n
0,4000,37500.0,0.086837
1,8000,18750.0,0.171389
2,12000,12500.0,0.253716
3,16000,9375.0,0.333877


## FPB Sallen Key
- Orden 2
- Ganancia N
- Freq corte: 2k
- Freq atenuacion: 8k
- $A = \frac{fc}{fat}$

In [ ]:
fc = 720
rn = 4.7e3
H0 = 1
a = np.sqrt(2)
b = 1

def calculate_filter_components(H0, a = np.sqrt(2), b = 1):
    r1n = r2n = r3n = 1
    r4n = H0 - 1
    c1n = (a + np.sqrt(a**2 + 8 * b * (H0 - 1))) / (4 * b)
    c2n = 4 / (a + np.sqrt(a**2 + 8 * b * (H0 - 1)))
    return r1n, r2n, r3n, r4n, c1n, c2n

normal_values = calculate_filter_components(H0, a, b)
df = pd.DataFrame([normal_values], columns=["R1n", "R2n", "R3n", "R4n", "C1n", "C2n"])
df

# c1 = normal_values[4] / (2 * np.pi * fc * rn)
# c2 = normal_values[5] / (2 * np.pi * fc * rn)


# print(f"R1: {r1n * rn / 1e3:.2f} kΩ, R4: {r4n * rn / 1e3:.2f} kΩ")
# print(f"C1: {c1 / 1e-9:.2f} nF, C2: {c2 / 1e-9:.2f} nF")

,R1n,R2n,R3n,R4n,C1n,C2n
0,1,1,1,0,0.707107,1.414214


In [11]:
## Para ganacia 1
a = np.sqrt(2)
b = 1

c1n = a / (2*b)
c2n = 2 / a
r_values = [1e3, 2.2e3, 3.3e3, 4.7e3, 5.1e3, 6.8e3, 8.2e3]
test_values = [
    (r, c1n, c2n) for r in r_values
]
df = pd.DataFrame(test_values, columns=["R", "C1", "C2"])
for freq in range(300, 1000, 100):
    df[f"C1[nF]_{freq}"] = (df["C1"] * 1e9 / (2 * np.pi * freq * df["R"])).round(2)
    df[f"C2[nF]_{freq}"] = (df["C2"] * 1e9 / (2 * np.pi * freq * df["R"])).round(2)
df

,R,C1,C2,C1[nF]_300,C2[nF]_300,C1[nF]_400,C2[nF]_400,C1[nF]_500,C2[nF]_500,C1[nF]_600,C2[nF]_600,C1[nF]_700,C2[nF]_700,C1[nF]_800,C2[nF]_800,C1[nF]_900,C2[nF]_900
0,1000.0,0.707107,1.414214,375.13,750.26,281.35,562.70,225.08,450.16,187.57,375.13,160.77,321.54,140.67,281.35,125.04,250.09
1,2200.0,0.707107,1.414214,170.51,341.03,127.89,255.77,102.31,204.62,85.26,170.51,73.08,146.16,63.94,127.89,56.84,113.68
2,3300.0,0.707107,1.414214,113.68,227.35,85.26,170.51,68.21,136.41,56.84,113.68,48.72,97.44,42.63,85.26,37.89,75.78
3,4700.0,0.707107,1.414214,79.82,159.63,59.86,119.72,47.89,95.78,39.91,79.82,34.21,68.41,29.93,59.86,26.61,53.21
4,5100.0,0.707107,1.414214,73.56,147.11,55.17,110.33,44.13,88.27,36.78,73.56,31.52,63.05,27.58,55.17,24.52,49.04
5,6800.0,0.707107,1.414214,55.17,110.33,41.37,82.75,33.10,66.20,27.58,55.17,23.64,47.29,20.69,41.37,18.39,36.78
6,8200.0,0.707107,1.414214,45.75,91.50,34.31,68.62,27.45,54.90,22.87,45.75,19.61,39.21,17.16,34.31,15.25,30.50


In [28]:
Vpwm_div = 10 / 32
df_pwm_resolution = pd.DataFrame([res for res in range(2000, 12100, 2000)], columns=["Resolution"])
df_pwm_resolution["Freq[kHz]"] = (150e6 / df_pwm_resolution["Resolution"]).round(2)
df_pwm_resolution["Sensibilidad[uV/bit]"] = (3.3 * Vpwm_div * 1e6 / df_pwm_resolution["Resolution"]).round(2)
df_pwm_resolution

,Resolution,Freq[kHz],Sensibilidad[uV/bit]
0,2000,75000.0,515.62
1,4000,37500.0,257.81
2,6000,25000.0,171.88
3,8000,18750.0,128.91
4,10000,15000.0,103.12
5,12000,12500.0,85.94
